# MMBSA_200 — Residue-Fill Review via PyMOL

**Runs on CESGA via `juplaunch`**  (memory `[[reference_juplaunch]]`).

## Connection setup

On your **laptop** before launching this notebook:

```bash
# 1. Start PyMOL with the RPC server on the laptop
export PYMOL_PATH=/usr/lib/python3/dist-packages/pymol   # Debian-fix per feedback_pymol_remote_debian
pymol -R
# → PyMOL RPC listening on :9123

# 2. Set up the reverse tunnel to CESGA
ssh -R 9123:localhost:9123 ft3.cesga.es
```

On CESGA:

```bash
~/bin/juplaunch     # prints the JupyterLab URL to paste into your laptop browser
```

Then open this notebook in the browser and run the cells top-down.
The `PymolSession(hostname="localhost", port=9123)` call reaches through
the reverse tunnel to the laptop's PyMOL GUI.


## Step 1 — Connect to laptop PyMOL

In [1]:
# Connect to laptop PyMOL via reverse tunnel (localhost:9123)
from pymol_remote.client import PymolSession
import sys, pathlib

pm = PymolSession(hostname="localhost", port=9123, timeout=10.0)
print("Connected. Objects currently loaded:", pm.get_names())


ImportError: cannot import name 'cached_property' from 'functools' (/mnt/netapp1/Optcesga_FT2_RHEL7/2020/gentoo/22072020/usr/lib/python3.7/functools.py)

## Step 2 — Discover MMBSA_200 pairs (crystal + filled)

In [ ]:
# Discover MMBSA_200 proteins with available filled models
import pathlib, glob

CRYSTAL_ROOT = pathlib.Path("/mnt/netapp1/Store_othcxlwa/FRUTON-NEW")
FILLED_GLOBS = [
    pathlib.Path("/mnt/lustre/scratch/nlsas/home/otras/hcx/lwa/MMBSA_200/bench_20260823_1620/artefacts"),
    pathlib.Path("/mnt/lustre/scratch/nlsas/home/otras/hcx/lwa/MMBSA_200/bench_20260823_1513/artefacts"),
]

crystal_pdbs = sorted(p.name for p in CRYSTAL_ROOT.iterdir() if p.is_dir())
filled_pdbs = set()
for fg in FILLED_GLOBS:
    if fg.is_dir():
        for f in fg.glob("*_final.pdb"):
            filled_pdbs.add(f.stem.replace("_final", ""))

pairs = sorted(p for p in crystal_pdbs if p in filled_pdbs)
print(f"{len(crystal_pdbs)} crystals total; {len(filled_pdbs)} have filled models; "
      f"{len(pairs)} pairs available for review.")
print("Pairs:", " ".join(pairs))


## Step 3 — Loader

MMBSA_200 does not (yet) ship a graft-manifest listing gap ranges, so filled-region residues are inferred as anything in the filled model that isn't within 0.5 Å of a crystal atom.

In [ ]:
# Loader for MMBSA_200: crystal from FRUTON-NEW/<PDB>/<PDB>.pdb,
# filled from SLURM bench_20260823_1620 (iter-4) or _1513 (iter-3) if newer.
import pathlib, json

def _pick_filled(pdb_id: str) -> pathlib.Path | None:
    for fg in FILLED_GLOBS:
        p = fg / f"{pdb_id}_final.pdb"
        if p.is_file():
            return p
    return None


def load_mmbsa200_fill(pdb_id: str) -> dict:
    return {
        "pdb_id": pdb_id,
        "crystal": (CRYSTAL_ROOT / pdb_id / f"{pdb_id}.pdb")
                   if (CRYSTAL_ROOT / pdb_id / f"{pdb_id}.pdb").is_file() else None,
        "filled": _pick_filled(pdb_id),
    }


def pymol_show_mmbsa_fill(pdb_id: str, gray_crystal: bool = True) -> dict:
    r = load_mmbsa200_fill(pdb_id)
    pm.do("reinitialize")
    pm.do("bg_color white")
    if r["crystal"]:
        pm.do(f"load {r['crystal']}, crystal")
        pm.do(("color grey70, crystal" if gray_crystal else "color yellow, crystal"))
        pm.do("show cartoon, crystal")
    if r["filled"]:
        pm.do(f"load {r['filled']}, filled")
        pm.do("color skyblue, filled")
        pm.do("show cartoon, filled")
        # Highlight residues present in filled but absent in crystal
        pm.do("select gap_fill, filled and not (byres filled within 0.5 of crystal)")
        pm.do("color hotpink, gap_fill")
        pm.do("show sticks, gap_fill and (name CA+CB+N+C+O)")
    if r["crystal"] and r["filled"]:
        pm.do("align filled and name CA, crystal and name CA")
    pm.do("orient")
    pm.do("zoom")
    print(f"{pdb_id}: crystal={bool(r['crystal'])}, filled={bool(r['filled'])}")
    return r


## Step 4 — Interactive selector

In [ ]:
import ipywidgets as W
from IPython.display import display

if not pairs:
    print("No filled models found — check FILLED_GLOBS paths.")
else:
    dropdown = W.Dropdown(options=pairs, description="PDB:", layout={"width": "260px"})
    gray_toggle = W.Checkbox(value=True, description="Grey crystal")
    btn = W.Button(description="Show in PyMOL", button_style="primary")
    out = W.Output()

    def _on_click(_):
        with out:
            out.clear_output()
            pymol_show_mmbsa_fill(dropdown.value, gray_crystal=gray_toggle.value)

    btn.on_click(_on_click)
    display(W.HBox([dropdown, gray_toggle, btn]), out)
